In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no")

True Tesla T4


In [10]:
!pip install -q transformers datasets scikit-learn

In [11]:
import pandas as pd
train_df = pd.read_csv("/content/drive/MyDrive/contract-ai/train.csv")
val_df = pd.read_csv("/content/drive/MyDrive/contract-ai/val.csv")
print(train_df.shape, val_df.shape)

(10953, 4) (1253, 4)


In [12]:
import pandas as pd
train_df = pd.read_csv("/content/drive/MyDrive/contract-ai/train.csv")
print(train_df.shape)
train_df.head()

(10953, 4)


,contract,clause_text,label,all_labels
0,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT,"""Provider""",Parties,Parties
1,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT,Ability Computer & Software Industries Ltd,Parties,Parties
2,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT,"All writings or works of authorship, including...",Ip Ownership Assignment,Ip Ownership Assignment
3,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT,"Each of the Recipient and the Provider may, in...",Termination For Convenience,Termination For Convenience
4,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT,Each of the foregoing parties is referred to h...,Parties,Parties


In [13]:
import json

label_list = sorted(train_df["label"].unique())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(len(label_list), "labels")
print(label2id["Other"])

42 labels
28


In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "lexlms/legal-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: lexlms/legal-roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
from datasets import Dataset

def tokenize(batch):
    return tokenizer(
        batch["clause_text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

train_df["labels"] = train_df["label"].map(label2id)
val_df["labels"] = val_df["label"].map(label2id)

train_ds = Dataset.from_pandas(train_df[["clause_text", "labels"]])
val_ds = Dataset.from_pandas(val_df[["clause_text", "labels"]])

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(train_ds[0].keys())

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Map:   0%|          | 0/10953 [00:00<?, ? examples/s]

Map:   0%|          | 0/1253 [00:00<?, ? examples/s]

dict_keys(['labels', 'input_ids', 'attention_mask'])


In [16]:
import json
import torch

with open("/content/drive/MyDrive/contract-ai/class_weights.json") as f:
    weights_by_label = json.load(f)

# must be in the same order as label_list / id2label
weight_list = [weights_by_label[id2label[i]] for i in range(len(label_list))]
class_weights = torch.tensor(weight_list, dtype=torch.float)
print(class_weights[:5])

tensor([4.2062, 5.4330, 1.4251, 0.5344, 0.4750])


In [17]:
import os
os.listdir("/content/drive/MyDrive/contract-ai")

['train.csv', 'val.csv', 'class_weights.json', 'test.csv']

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import os
os.listdir("/content/drive/MyDrive")

['Colab Notebooks',
 'Getting started.pdf',
 'Classroom',
 'IMG_20201103_194707.jpg',
 'IMG_20201103_194715 (2).jpg',
 'IMG_20201103_194715 (1).jpg',
 'IMG_20201103_194715.jpg',
 'IMG_20201103_194719 (1).jpg',
 'IMG_20201103_194719.jpg',
 'IMG_20201105_195930.jpg',
 'IMG_20201105_195946.jpg',
 'IMG_20201105_195938.jpg',
 'IMG_20201107_164527.jpg',
 'IMG_20201107_192623.jpg',
 'IMG_20201110_174047.jpg',
 'IMG_20201110_174051.jpg',
 'IMG_20201110_190547 (1).jpg',
 'IMG_20201110_190547.jpg',
 'IMG_20201110_190550 (1).jpg',
 'IMG_20201110_190550.jpg',
 'IMG_20201112_174702.jpg',
 'IMG_20201112_174706.jpg',
 'IMG_20201112_174708.jpg',
 'IMG_20201112_174712 (1).jpg',
 'IMG_20201112_174712.jpg',
 'IMG_20201113_162239 (2).jpg',
 'IMG_20201113_162245 (2).jpg',
 'IMG_20201113_162254 (2).jpg',
 'IMG_20201113_162259 (2).jpg',
 'IMG_20201113_162302 (1).jpg',
 'IMG_20201113_162239 (1).jpg',
 'IMG_20201113_162245 (1).jpg',
 'IMG_20201113_162254 (1).jpg',
 'IMG_20201113_162259 (1).jpg',
 'IMG_20201113

In [20]:
os.listdir("/content/drive/MyDrive/contract-ai")

['train.csv', 'val.csv', 'class_weights.json', 'test.csv']

In [21]:
import json
import torch

with open("/content/drive/MyDrive/contract-ai/class_weights.json") as f:
    weights_by_label = json.load(f)

# must be in the same order as label_list / id2label
weight_list = [weights_by_label[id2label[i]] for i in range(len(label_list))]
class_weights = torch.tensor(weight_list, dtype=torch.float)
print(class_weights[:5])

tensor([4.2062, 5.4330, 1.4251, 0.5344, 0.4750])


In [22]:
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [23]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1_macro": f1, "precision_macro": precision, "recall_macro": recall}

In [24]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/contract-ai/model_checkpoints",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    fp16=True,
    report_to="none",
)

In [25]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,1.015756,0.911193,0.811652,0.675816,0.675983,0.750064
2,0.593926,0.874204,0.826816,0.701521,0.696194,0.753916
3,0.376130,0.898993,0.834796,0.693057,0.689295,0.725737
4,0.254584,0.895637,0.836393,0.704374,0.703624,0.725900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2740, training_loss=0.7515020836878867, metrics={'train_runtime': 602.5121, 'train_samples_per_second': 72.716, 'train_steps_per_second': 4.548, 'total_flos': 5765780780052480.0, 'train_loss': 0.7515020836878867, 'epoch': 4.0})

In [26]:
trainer.save_model("/content/drive/MyDrive/contract-ai/final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]